# Comparación final de modelos — HistGradientBoosting vs. Random Forest

**Continuación de:** `03-modelo-predictivo.ipynb` y `04-modelado-lineal-y-random-forest.ipynb`

Cada notebook entrenó y comparó sus propios candidatos por separado, y cada uno exportó su propio "ganador":

- `03-modelo-predictivo.ipynb` → `HistGradientBoostingRegressor` → `models/modelo_hist_gradient_boosting.joblib`
- `04-modelado-lineal-y-random-forest.ipynb` → `RandomForestRegressor` (sin `pasajeros`) → `models/modelo_random_forest.joblib`

No se compararon esos dos ganadores entre sí, bajo las mismas condiciones. Eso es lo que hace este notebook: **carga los dos modelos ya entrenados** (no se reentrena nada), los evalúa con el mismo `test.csv.gz` y el mismo código de métricas, y declara **un único modelo final para la Fase 1**.

## Configuración e imports

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_PATH = "../data/processed"
MODELS_DIR = "../models"
OBJETIVO = "duracion_viaje_horas"

RUTA_HGB = os.path.join(MODELS_DIR, "modelo_hist_gradient_boosting.joblib")
RUTA_RF = os.path.join(MODELS_DIR, "modelo_random_forest.joblib")

## 1. Cargar los datos de prueba y los dos modelos

Se usa **únicamente el conjunto de prueba** (`test.csv.gz`): los dos modelos ya están entrenados, aquí solo se evalúan. No hay riesgo de fuga de información porque no se ajusta nada nuevo con estos datos.

In [2]:
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv.gz"))
y_test = test[OBJETIVO]
mask_sin_corregir = (~test["duracion_corregida"]).to_numpy()

modelo_hgb = joblib.load(RUTA_HGB)
modelo_rf = joblib.load(RUTA_RF)

print(f"Filas de prueba: {len(test):,} (sin corregir: {mask_sin_corregir.sum():,})")
print(f"Tamaño en disco -> HistGradientBoosting: {os.path.getsize(RUTA_HGB) / 1e6:.2f} MB "
      f"| Random Forest: {os.path.getsize(RUTA_RF) / 1e6:.2f} MB")

Filas de prueba: 523,425 (sin corregir: 522,712)
Tamaño en disco -> HistGradientBoosting: 0.27 MB | Random Forest: 47.38 MB


## 2. Preparar las predictoras para cada modelo

Los dos modelos **no** se entrenaron con el mismo formato de entrada, así que no se les puede pasar el mismo `X_test` tal cual:

- **Random Forest** es un `Pipeline` completo (incluye su propio `ColumnTransformer`): recibe las columnas originales, en texto, y las codifica él mismo. Como usa `remainder="drop"`, no importa si el DataFrame trae columnas de más.
- **HistGradientBoosting** es el estimador solo, sin `Pipeline`, usando el soporte nativo de categóricas de scikit-learn. Esto exige exactamente las **9 columnas con las que se entrenó** (ni una más ni una menos) y que las 5 categóricas vengan como `dtype="category"` — igual que se preparó en `03-modelo-predictivo.ipynb`. Pasarle otro formato lo haría fallar o predecir mal.

Por eso se arman dos versiones de las predictoras a partir del mismo `test`.

In [3]:
CATEGORICAS = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo"]

# --- Para Random Forest (notebook 04): columnas originales + hora_entera, su Pipeline selecciona lo que necesita ---
test_rf = test.copy()
test_rf["hora_entera"] = test_rf["hora_salida_decimal"].astype(int)
PREDICTORAS_RF = CATEGORICAS + ["pasajeros", "hora_salida_decimal", "dia_semana", "mes", "hora_entera"]
X_test_rf = test_rf[PREDICTORAS_RF]

# --- Para HistGradientBoosting (notebook 03): exactamente sus 9 columnas, categoricas como category ---
PREDICTORAS_HGB = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n",
                    "clase_veh_culo", "pasajeros", "hora_salida_decimal", "dia_semana", "mes"]
X_test_hgb = test[PREDICTORAS_HGB].copy()
for c in CATEGORICAS:
    X_test_hgb[c] = X_test_hgb[c].astype("category")

print("Columnas para Random Forest:", list(X_test_rf.columns))
print("Columnas para HistGradientBoosting:", list(X_test_hgb.columns))

Columnas para Random Forest: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'hora_salida_decimal', 'dia_semana', 'mes', 'hora_entera']
Columnas para HistGradientBoosting: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'hora_salida_decimal', 'dia_semana', 'mes']


## 3. Predecir y evaluar con el mismo criterio

Mismas métricas para los dos, calculadas con el mismo código: **MAE** (métrica principal, en horas), **RMSE** y **R²**, sobre el conjunto de prueba completo y también solo sobre las filas con duración real (`duracion_corregida == False`), para confirmar que ningún resultado depende de las duraciones que se reemplazaron en la preparación (notebook 02).

También se mide el **tiempo de predicción** sobre todo el conjunto de prueba: con métricas tan parecidas, la velocidad y el tamaño del archivo pueden ser el criterio que desempate.

In [4]:
def evaluar(nombre, modelo, X):
    t0 = time.time()
    pred = modelo.predict(X)
    seg = time.time() - t0

    y_real, y_c = y_test.to_numpy(), y_test.to_numpy()[mask_sin_corregir]
    p, p_c = pred, pred[mask_sin_corregir]

    return {
        "Modelo": nombre,
        "MAE (Global)": mean_absolute_error(y_real, p),
        "RMSE (Global)": np.sqrt(mean_squared_error(y_real, p)),
        "R2 (Global)": r2_score(y_real, p),
        "MAE (Sin Corregir)": mean_absolute_error(y_c, p_c),
        "RMSE (Sin Corregir)": np.sqrt(mean_squared_error(y_c, p_c)),
        "R2 (Sin Corregir)": r2_score(y_c, p_c),
        "Tiempo de prediccion (s)": seg,
        "Tamaño en disco (MB)": os.path.getsize(RUTA_HGB if nombre == "HistGradientBoosting" else RUTA_RF) / 1e6,
    }

resultados = pd.DataFrame([
    evaluar("HistGradientBoosting", modelo_hgb, X_test_hgb),
    evaluar("Random Forest", modelo_rf, X_test_rf),
]).set_index("Modelo")

resultados.round(4)

,MAE (Global),RMSE (Global),R2 (Global),MAE (Sin Corregir),RMSE (Sin Corregir),R2 (Sin Corregir),Tiempo de prediccion (s),Tamaño en disco (MB)
Modelo,,,,,,,,
HistGradientBoosting,0.3899,0.7852,0.9523,0.3893,0.7841,0.9523,2.3445,0.2660
Random Forest,0.3622,0.7871,0.9520,0.3616,0.7861,0.9520,3.4171,47.3774


## 4. Resultado y conclusión

In [5]:
ganador = resultados["MAE (Global)"].idxmin()
perdedor = [m for m in resultados.index if m != ganador][0]

dif_mae = (resultados.loc[perdedor, "MAE (Global)"] - resultados.loc[ganador, "MAE (Global)"])
dif_mae_pct = dif_mae / resultados.loc[perdedor, "MAE (Global)"] * 100

print(f"Ganador por MAE (métrica principal): {ganador}")
print(f"MAE {ganador}: {resultados.loc[ganador, 'MAE (Global)']:.4f} h "
      f"vs. MAE {perdedor}: {resultados.loc[perdedor, 'MAE (Global)']:.4f} h "
      f"({dif_mae_pct:.1f}% menor, {dif_mae * 60:.1f} minutos de diferencia)")
print(f"RMSE {ganador}: {resultados.loc[ganador, 'RMSE (Global)']:.4f} h "
      f"vs. RMSE {perdedor}: {resultados.loc[perdedor, 'RMSE (Global)']:.4f} h")
print(f"R2 {ganador}: {resultados.loc[ganador, 'R2 (Global)']:.4f} "
      f"vs. R2 {perdedor}: {resultados.loc[perdedor, 'R2 (Global)']:.4f}")
print(f"\nTamaño en disco -> {ganador}: {resultados.loc[ganador, 'Tamaño en disco (MB)']:.2f} MB "
      f"| {perdedor}: {resultados.loc[perdedor, 'Tamaño en disco (MB)']:.2f} MB")
print(f"Tiempo de predicción (todo el test) -> {ganador}: {resultados.loc[ganador, 'Tiempo de prediccion (s)']:.2f} s "
      f"| {perdedor}: {resultados.loc[perdedor, 'Tiempo de prediccion (s)']:.2f} s")

Ganador por MAE (métrica principal): Random Forest
MAE Random Forest: 0.3622 h vs. MAE HistGradientBoosting: 0.3899 h (7.1% menor, 1.7 minutos de diferencia)
RMSE Random Forest: 0.7871 h vs. RMSE HistGradientBoosting: 0.7852 h
R2 Random Forest: 0.9520 vs. R2 HistGradientBoosting: 0.9523

Tamaño en disco -> Random Forest: 47.38 MB | HistGradientBoosting: 0.27 MB
Tiempo de predicción (todo el test) -> Random Forest: 3.42 s | HistGradientBoosting: 2.34 s


## Conclusión

**Modelo oficial de la Fase 1: Random Forest**, por tener el MAE más bajo de los dos — 0,362 h frente a 0,390 h de HistGradientBoosting, un 7,1 % menos (≈ 1,7 minutos de diferencia por viaje). El MAE es la métrica principal del proyecto (ver `fase-1/README.md`), así que es el criterio que decide.

En RMSE y R² los dos modelos quedan prácticamente empatados (RMSE 0,787 h vs. 0,785 h; R² 0,9520 vs. 0,9523; diferencias de 0,002 h y 0,0003 respectivamente), sin cambiar la conclusión: ambos explican alrededor del 95 % de la variación de la duración del viaje. Tampoco hay contradicción con el resultado sobre las filas sin corregir (`duracion_corregida == False`): en los dos modelos es prácticamente idéntico al global, así que ninguno depende de las duraciones que se reemplazaron en la preparación (notebook 02).

HistGradientBoosting queda documentado como alternativa, no como candidato descartado: es ~175 veces más liviano (0,27 MB frente a 47,38 MB) y algo más rápido al predecir (2,34 s frente a 3,42 s sobre todo el test), una diferencia que puede ser relevante para el despliegue en las fases siguientes del proyecto aunque no cambie la elección de la Fase 1.
